# 001 - Bronze Ingest

This notebook ingests raw data files from the workspace into the **bronze** layer of the `rentalproperty` catalog.

The databricks CATALOG and schema's are created as well if it is not existing yet in the databricks workspace

**Sources:**
* `airbnb.csv` — Airbnb listing data
* `rentals.json` — Rental listing data

**Targets:**
* `rentalproperty.bronze.airbnb`
* `rentalproperty.bronze.rentals`

Files are read via pandas (workspace path workaround for serverless) and written as managed Delta tables using overwrite mode.

In [0]:
#set all variables to be used in this notebook
catalog_name = "rentalproperty"
schema_name_bronze = "bronze"
schema_name_silver = "silver"
schema_name_gold = "gold"
airbnb_input_csv_path = "/Workspace/Repos/airbnb/rent-airbnb/data/input/airbnb.csv"
rentals_input_csv_path = "/Workspace/Repos/airbnb/rent-airbnb/data/input/rentals.json"

airbnb_table_name = "airbnb"
rentals_table_name = "rentals"

airbnb_table_name_full = f"{catalog_name}.{schema_name_bronze}.{airbnb_table_name}"
rentals_table_name_full = f"{catalog_name}.{schema_name_bronze}.{rentals_table_name}"


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name_bronze}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name_silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name_gold}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField,StringType, DoubleType, IntegerType, BooleanType, ArrayType)

#load airnbnb csv file into dataframe
df_airbnb = spark.read.format("csv").option("header","true").option("inferSchema","true").load(airbnb_input_csv_path)
df_airbnb = df_airbnb.withColumn("ingestedDateTime", F.current_timestamp())

#ingest airnbnb dataset using databricks managed table 
df_airbnb.write.mode("overwrite").saveAsTable(airbnb_table_name_full)

# Schema definition for rentals as large json file could give indifferent results on ingestion
rentals_schema = StructType([
    StructField("_id", ArrayType(StringType()), True),
    StructField("additionalCostsRaw", StringType(), True),
    StructField("areaSqm", StringType(), True),
    StructField("city", StringType(), True),
    StructField("crawlStatus", StringType(), True),
    StructField("crawledAt", ArrayType(StringType()), True),
    StructField("deposit", StringType(), True),
    StructField("descriptionTranslated", StringType(), True),
    StructField("detailsCrawledAt", ArrayType(StringType()), True),
    StructField("energyLabel", StringType(), True),
    StructField("firstSeenAt", ArrayType(StringType()), True),
    StructField("furnish", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("internet", StringType(), True),
    StructField("isRoomActive", StringType(), True),
    StructField("kitchen", StringType(), True),
    StructField("lastSeenAt", ArrayType(StringType()), True),
    StructField("latitude", DoubleType(), True),
    StructField("living", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("matchCapacity", StringType(), True),
    StructField("pageDescription", StringType(), True),
    StructField("pageTitle", StringType(), True),
    StructField("pets", StringType(), True),
    StructField("postalCode", StringType(), True),
    StructField("postedAgo", StringType(), True),
    StructField("propertyType", StringType(), True),
    StructField("availability", StringType(), True),
    StructField("registrationCost", StringType(), True),
    StructField("rent", StringType(), True),
    StructField("roommates", IntegerType(), True),
    StructField("shower", StringType(), True),
    StructField("smokingInside", StringType(), True),
    StructField("source", StringType(), True),
    StructField("title", StringType(), True),
    StructField("toilet", StringType(), True),
])

# Load rentals json using schema into dataframe
df_rentals = spark.read.schema(rentals_schema).json(rentals_input_csv_path)
df_rentals = df_rentals.withColumn("ingestedDateTime", F.current_timestamp())

#ingest rentals dataset using databricks managed table 
df_rentals.write.mode("overwrite").saveAsTable(rentals_table_name_full)